In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import skewnorm
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 20  # MB
#Set the limit higher if you want the entire animation to process

In [ ]:
rng = np.random.default_rng()

In [ ]:
def dist(L, x1, y1, x2, y2):
    
    dx = x2 - x1
    dx -= L*round(dx/L)
    dy = y2 - y1
    dy -= L*round(dy/L)
    
    return dx, dy

In [ ]:
def force_x(L, k_0, r_0, x1, y1, x2, y2):
    '''This function evaluates the horizontal force acting 
    on the particle at a position (x1, y1) due to another
    particle located at (x2, y2)'''
    dx, dy = dist(L, x1, y1, x2, y2)
    r = np.sqrt(dx*dx + dy*dy)
    if r > r_0 or r == 0:
        f_x = 0
    else:
        f_x = -k_0*(r_0 - r)*(dx)/r

    return f_x
     

In [ ]:
def force_y(L, k_0, r_0, x1, y1, x2, y2):
    '''This function evaluates the vertical force acting 
    on the particle at a position (x1, y1) due to another
    particle located at (x2, y2)'''
    dx, dy = dist(L, x1, y1, x2, y2)
    r = np.sqrt(dx*dx + dy*dy)
    if r > r_0 or r == 0:
        f_y = 0
    else:
        f_y = -k_0*(r_0 - r)*(dy)/r

    return f_y

In [ ]:
def particle_trajectory_m(P, k_0, r_0, k, w, D_r, L, delta_t, N, x_0, y_0, theta_0):
    
    x_grid = np.zeros((N, P))
    y_grid = np.zeros((N, P))
    theta_grid = np.zeros((N, P))
    for i in range(P):
        x_grid[0][i] = x_0[i]
        y_grid[0][i] = y_0[i]
        theta_grid[0][i] = theta_0[i]
    i = 1

    while i < N:
        for j in range(P):
            force_h = 0
            force_v = 0
            for l in range(P):
                force_h += force_x(L, k_0, r_0, x_grid[i-1][j], y_grid[i-1][j], x_grid[i-1][l], y_grid[i-1][l])
                force_v += force_y(L, k_0, r_0, x_grid[i-1][j], y_grid[i-1][j], x_grid[i-1][l], y_grid[i-1][l])
            x_grid[i][j] = x_grid[i-1][j] + np.cos(theta_grid[i-1][j])*delta_t + force_h*delta_t
            y_grid[i][j] = y_grid[i-1][j] + np.sin(theta_grid[i-1][j])*delta_t + force_v*delta_t

            d_beta = rng.normal(0, np.sqrt(delta_t))
            theta_grid[i][j] = (theta_grid[i-1][j] + (1 + w)*delta_t + force_h*np.cos(theta_grid[i-1][j])*delta_t +
                                                force_v*np.sin(theta_grid[i-1][j])*delta_t + np.sqrt(2*D_r)*d_beta)
            if x_grid[i][j] > L:
                x_grid[i][j] = 0 + (x_grid[i][j] - L*np.floor(x_grid[i][j]/L))
            elif x_grid[i][j] < 0:
                x_grid[i][j] = L + (x_grid[i][j] - L*(np.floor(x_grid[i][j]/L)+1))
            if y_grid[i][j] > L:
                y_grid[i][j] = 0 + (y_grid[i][j] - L*np.floor(y_grid[i][j]/L))
            elif y_grid[i][j] < 0:
                y_grid[i][j] = L + (y_grid[i][j] - L*(np.floor(y_grid[i][j]/L)+1))
        i += 1

    return x_grid, y_grid, theta_grid

In [ ]:
def func(K, P, k_0, r_0, k, w, D_r, L, delta_t, N, x0_arr, y0_arr, t0_arr):
        
    #Grids to store the co-ordinates of
    #P particles across N steps
    x_grid = np.zeros((N, P))
    y_grid = np.zeros((N, P))
    theta_grid = np.zeros((N, P))

    for i in range(P):
        x_grid[0][i] = x0_arr[i]
        y_grid[0][i] = y0_arr[i]
        theta_grid[0][i] = t0_arr[i]

    #The size of the bins
    l = L/K

    def build_cells(x, y):
        '''Discrete bins for storing particle indicies'''
        cells = [[[] for _ in range(K)] for _ in range(K)]
        for j in range(P):
            cx = int(x[j]/l)%K
            cy = int(y[j]/l)%K
            cells[cx][cy].append(j)
        return cells

    cells = build_cells(x_grid[0], y_grid[0])

    index_disp = [[0, 0], [1, 0], [-1, 0], [0, 1], [0, -1], [1, 1], [-1, 1], [-1, -1], [1, -1]]

    i = 1

    while i < N:
        for j in range(P):
            force_h = 0
            force_v = 0
            x_index = int(np.floor(x_grid[i-1][j]/l))
            y_index = int(np.floor(y_grid[i-1][j]/l))
            for n in range(9):
                for m in cells[(x_index + index_disp[n][0])%K][(y_index + index_disp[n][1])%K]:
                    force_h += force_x(L, k_0, r_0, x_grid[i-1][j], y_grid[i-1][j], x_grid[i-1][m], y_grid[i-1][m])
                    force_v += force_y(L, k_0, r_0, x_grid[i-1][j], y_grid[i-1][j], x_grid[i-1][m], y_grid[i-1][m])
                        
            x_grid[i][j] = x_grid[i-1][j] + np.cos(theta_grid[i-1][j])*delta_t + force_h*delta_t
            y_grid[i][j] = y_grid[i-1][j] + np.sin(theta_grid[i-1][j])*delta_t + force_v*delta_t

            d_beta = rng.normal(0, np.sqrt(delta_t))
            theta_grid[i][j] = (theta_grid[i-1][j] + (1 + w)*delta_t + force_h*np.cos(theta_grid[i-1][j])*delta_t +
                                                force_v*np.sin(theta_grid[i-1][j])*delta_t + np.sqrt(2*D_r)*d_beta)
            if x_grid[i][j] >= L:
                x_grid[i][j] = 0 + (x_grid[i][j] - L*np.floor(x_grid[i][j]/L))
            elif x_grid[i][j] < 0:
                x_grid[i][j] = L + (x_grid[i][j] - L*(np.floor(x_grid[i][j]/L)+1))
            if y_grid[i][j] >= L:
                y_grid[i][j] = 0 + (y_grid[i][j] - L*np.floor(y_grid[i][j]/L))
            elif y_grid[i][j] < 0:
                y_grid[i][j] = L + (y_grid[i][j] - L*(np.floor(y_grid[i][j]/L)+1))

        cells = build_cells(x_grid[i], y_grid[i])

        i += 1

    return x_grid, y_grid, theta_grid

In [ ]:
x_in = np.ones(10000) * 1.0
y_in = np.zeros(10000)
t_in = np.zeros(10000)

for i in range(10000):
    x_in[i] = rng.uniform(1, 99)
    y_in[i] = rng.uniform(1, 99)
    t_in[i] = rng.normal(1)


x2, y2, t2 = func(99, 5500, 50, 1.0, 15, -0.65, 0.05, 100.0, 0.01, 12000, x_in, y_in, t_in)

In [ ]:
fig_2D = plt.figure(figsize=(10,6),tight_layout=True)
index = 1
for i in range(15):
    ax = fig_2D.add_subplot(3, 5, index)
    #ax.plot(x[200 + 2*i], y[200 + 2*i], '.')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'Step {800*i}')
    xi = x2[800*i]
    yi = y2[800*i]
    dx = np.cos(t2[800*i])
    dy = np.sin(t2[800*i])

    # Add arrows
    # create colour array
    P = len(xi)
    colors = ['red']*0 + ['black']*(P-1)

    # Add arrows with different colours
    ax.quiver(xi, yi, dx, dy,
              angles='xy',
              scale_units='xy',
              scale=5,
              width=0.008,
              color=colors)

    index += 1

#plt.savefig('MIPS_concrete')

In [ ]:
x_in = np.ones(3000) * 1.0
y_in = np.zeros(3000)
t_in = np.zeros(3000)

for i in range(3000):
    x_in[i] = rng.normal(20)
    y_in[i] = rng.normal(20)
    t_in[i] = rng.normal(1)
    
x1000, y1000, t1000 = particle_trajectory_m(500, 15, 7.0, 15, -0.65, 0.05, 40.0, 0.01, 50, x_in, y_in, t_in)

In [ ]:
fig_2D = plt.figure(figsize=(10,6),tight_layout=True)
index = 1
for i in range(15):
    ax = fig_2D.add_subplot(3, 5, index)
    #ax.plot(x[200 + 2*i], y[200 + 2*i], '.')
    ax.set_xlabel('x')
    ax.set_ylabel(r"$\theta$")
    #ax.set_title(f'Wave after {i*dt}s')
    xi = x1000[650 + 20*i]
    yi = y1000[650 + 20*i]
    dx = np.cos(t1000[650 + 20*i])
    dy = np.sin(t1000[650 + 20*i])

    # Add arrows
    # create colour array
    P = len(xi)
    colors = ['red']*1 + ['black']*(P-1)

    # Add arrows with different colours
    ax.quiver(xi, yi, dx, dy,
              angles='xy',
              scale_units='xy',
              scale=5,
              width=0.02,
              color=colors)

    index += 1

In [ ]:



# -------------------------------------------------
# ANIMATION SETTINGS
# -------------------------------------------------
frame_step = 50      # skip time steps to make animation faster
arrow_step = 1       # draw orientation arrows for every 5th particle
arrow_length = 0.08  # visible arrow length in data units

# -------------------------------------------------
# CREATE FIGURE
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 7))
ax.set_xlim(0, 100.0)   # here L = 2.0 in your simulation call
ax.set_ylim(0, 100.0)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')

# show ALL particles
scat = ax.scatter(x2[0, :], y2[0, :], s=12)

# show SOME arrows to avoid clutter
u0 = arrow_length * np.cos(t2[0, ::arrow_step])
v0 = arrow_length * np.sin(t2[0, ::arrow_step])

quiv = ax.quiver(
    x2[0, ::arrow_step],
    y2[0, ::arrow_step],
    u0, v0,
    angles='xy',
    scale_units='xy',
    scale=1,
    width=0.015,
    headwidth=4,
    headlength=5,
    headaxislength=4
)

title = ax.set_title("time step = 0")

# -------------------------------------------------
# UPDATE FUNCTION
# -------------------------------------------------
def update(frame):
    # update ALL particle positions
    offsets = np.column_stack((x2[frame, :], y2[frame, :]))
    scat.set_offsets(offsets)

    # update arrow positions and directions
    arrow_offsets = np.column_stack((x2[frame, ::arrow_step], y2[frame, ::arrow_step]))
    u = arrow_length * np.cos(t2[frame, ::arrow_step])
    v = arrow_length * np.sin(t2[frame, ::arrow_step])

    quiv.set_offsets(arrow_offsets)
    quiv.set_UVC(u, v)

    title.set_text(f"time step = {frame}")
    return scat, quiv, title

frames = range(0, x2.shape[0], frame_step)

ani = FuncAnimation(
    fig,
    update,
    frames=frames,
    interval=200,
    blit=False,
    repeat=True
)

# -------------------------------------------------
# DISPLAY ANIMATION
# -------------------------------------------------
plt.close(fig)  # prevents a duplicate empty static plot in notebooks
display(HTML(ani.to_jshtml()))